# Baseline BM25 --- Notebook de apoio

**Disciplina:** Inteligência Artificial --- FACOM/UFMS --- 2026/1

Este notebook carrega o `corpus.jsonl` gerado pelo notebook de coleta (`coleta_arxiv.ipynb`), treina um índice **BM25** e devolve o *top-k* para uma consulta-exemplo. É o seu **baseline obrigatório**. A partir daqui, você só precisa adicionar pré-processamento mais cuidadoso, comparar com KNN/denso e implementar os módulos de aprofundamento.

**Atenção:** este código é propositalmente "mínimo viável". Você é avaliado também pelo **rigor das suas escolhas** (justificativa dos hiperparâmetros, pré-processamento, conexão com o paradigma probabilístico de Naïve Bayes). Não entregue isto como está --- evolua-o.

In [1]:
# !pip install rank_bm25 nltk pandas

In [2]:
import json
import sys
from pathlib import Path

import pandas as pd
from rank_bm25 import BM25Okapi

sys.path.append("..")
from src.preprocessing import preprocess  # mesmo pré-processamento usado pelo KNN/TF-IDF

## 1. Carregamento do corpus

Ajuste o caminho se o seu `corpus.jsonl` estiver em outro lugar.

In [3]:
CORPUS_PATH = Path("../data/corpus.jsonl")  # ajuste se necessário

docs = []
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        docs.append(json.loads(line))

print(f"Documentos carregados: {len(docs)}")
docs[0]

Documentos carregados: 1097


{'arxiv_id': '0807.3908',
 'title': 'A Distributed Process Infrastructure for a Distributed Data Structure',
 'abstract': 'The Resource Description Framework (RDF) is continuing to grow outside the bounds of its initial function as a metadata framework and into the domain of general-purpose data modeling. This expansion has been facilitated by the continued increase in the capacity and speed of RDF database repositories known as triple-stores. High-end RDF triple-stores can hold and process on the order of 10 billion triples. In an effort to provide a seamless integration of the data contained in RDF repositories, the Linked Data community is providing specifications for linking RDF data sets into a universal distributed graph that can be traversed by both man and machine. While the seamless integration of RDF data sets is important, at the scale of the data sets that currently exist and will ultimately grow to become, the "download and index" philosophy of the World Wide Web will not 

## 2. Pré-processamento

Usamos o mesmo pipeline de `src/preprocessing.py` (lower-casing, remoção de pontuação,
remoção de *stopwords*, stemming via Porter Stemmer) aplicado tanto à coleção quanto às
consultas — garante que os dois lados sejam comparáveis (etapa b do enunciado).

In [4]:
def tokenize(text: str):
    return preprocess(text)

tokenize("Self-Supervised Graph Neural Networks for Subgraph Matching in Property Graphs")

['selfsupervis',
 'graph',
 'neural',
 'network',
 'subgraph',
 'match',
 'properti',
 'graph']

In [5]:
# Texto indexado = título + abstract
texts = [(d["title"] + ". " + d["abstract"]) for d in docs]
tokenized_corpus = [tokenize(t) for t in texts]
print("Exemplo de tokens do doc 0:", tokenized_corpus[0][:20])

Exemplo de tokens do doc 0: ['distribut', 'process', 'infrastructur', 'distribut', 'data', 'structur', 'resourc', 'descript', 'framework', 'rdf', 'continu', 'grow', 'outsid', 'bound', 'initi', 'function', 'metadata', 'framework', 'domain', 'generalpurpos']


## 3. Índice BM25

Hiperparâmetros adotados: $k_1 = 1.5$ e $b = 0.75$.

- $k_1 = 1.5$: valor dentro da faixa típica (1.2–2.0); abstracts científicos são curtos e raramente
  repetem um termo de forma excessiva, então não há necessidade de saturar a contribuição do termo
  mais rapidamente (k1 menor).
- $b = 0.75$: padrão da literatura (Robertson & Zaragoza, 2009); títulos+abstracts da coleção têm
  comprimento relativamente homogêneo, então uma penalização moderada por tamanho é adequada.

Esses valores serão revisitados empiricamente no módulo M5 (ranking híbrido), que ajusta o peso de
combinação entre BM25 e o recuperador denso.

In [6]:
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)
print("Vocabulário BM25:", len(bm25.idf), "termos")

Vocabulário BM25: 9706 termos


## 4. Função de busca

In [7]:
def search(query: str, k: int = 10):
    q_tokens = tokenize(query)
    scores = bm25.get_scores(q_tokens)
    top_idx = scores.argsort()[::-1][:k]
    return [(int(i), float(scores[i]), docs[i]) for i in top_idx]

In [8]:
query = "subgraph matching algorithms for graph databases"
results = search(query, k=10)

for rank, (idx, score, d) in enumerate(results, 1):
    print(f"{rank:>2}. [{score:6.2f}] {d['title']}")
    print(f"     {d['arxiv_id']} | {d.get('primary_category')} | {d.get('published','')[:10]}")
    print(f"     {d['abstract'][:200]}...")
    print()

 1. [ 10.62] A customizable inexact subgraph matching algorithm for attributed graphs
     2512.04280 | cs.DS | 2025-12-03
     Graphs provide a natural way to represent data by encoding information about objects and the relationships between them. With the ever-increasing amount of data collected and generated, locating speci...

 2. [ 10.28] An Efficient Pruning Process with Locality Aware Exploration and Dynamic Graph Editing for Subgraph Matching
     2112.11736 | cs.IR | 2021-12-22
     Subgraph matching is a NP-complete problem that extracts isomorphic embeddings of a query graph $q$ in a data graph $G$. In this paper, we present a framework with three components: Preprocessing, Reo...

 3. [ 10.15] Efficient Subgraph Matching on Billion Node Graphs
     1205.6691 | cs.DB | 2012-05-30
     The ability to handle large scale graph data is crucial to an increasing number of applications. Much work has been dedicated to supporting basic graph operations such as subgraph matching, rea

## 5. Gerando um arquivo `runs/bm25.trec` para avaliação

Formato TREC tradicional: `qid Q0 doc_id rank score system`. Esse formato é aceito por `pytrec_eval` e `ir_measures`, e é o que o script `evaluate.py` (no diretório `eval/`) consome.

In [9]:
queries_path = Path("../eval/queries.tsv")  # qid<TAB>texto da query
runs_dir = Path("./runs"); runs_dir.mkdir(exist_ok=True)
run_path = runs_dir / "bm25.trec"

queries = pd.read_csv(queries_path, sep="\t", names=["qid", "text"])
with open(run_path, "w", encoding="utf-8") as f:
    for _, q in queries.iterrows():
        for rank, (idx, score, d) in enumerate(search(q["text"], k=100), 1):
            f.write(f"{q['qid']} Q0 {d['arxiv_id']} {rank} {score:.6f} bm25\n")

print("Run salva em:", run_path)
!head -n 5 {run_path}

Run salva em: runs/bm25.trec
q1 Q0 2512.04280 1 10.622529 bm25
q1 Q0 2112.11736 2 10.280303 bm25
q1 Q0 1205.6691 3 10.151638 bm25
q1 Q0 1206.6483 4 10.029544 bm25
q1 Q0 2201.11251 5 10.024241 bm25


## 6. Próximos passos

1. Substitua o tokenizador por algo mais robusto (e.g., spaCy) e compare.
2. Implemente um segundo *retriever* (KNN + TF-IDF, ou KNN + *embeddings*) e gere `runs/knn.trec`.
3. Construa as `qrels` (Seção 2 do template em `eval/qrels_example.tsv`).
4. Use `eval/evaluate.py` para comparar.
5. Implemente o(s) módulo(s) de aprofundamento e gere mais arquivos de *run*.
6. Escreva tudo no relatório SBC. ✍️